# PATH MANAGEMENT

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

%load_ext autoreload
%autoreload 2
# %matplotlib inline

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/notebooks
/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/app


In [ ]:
from src.config import Configuration

CONFIG = Configuration(
    batch_size=64,
    max_tok_length=16,

    num_shots=5,

    model_name="meta-llama/Meta-Llama-3-8B"
)

# Prompting

Prompting in LLMs is the design of a structured input to provide task description, demostrations and the actual input for the model to generate a desired output.

In this notebook, we are going to use for fine-tuning a dataset set that is already available in the [Datasets repository](https://huggingface.co/datasets) from Hugging Face. However, the [Datasets library](https://huggingface.co/docs/datasets) makes easy to access and load datasets. For example, you can easily load your own dataset following [this tutorial](https://huggingface.co/docs/datasets/loading#local-and-remote-files).

More precisely, we are going to explain how to perform In-Context Learning with the [Llama2 model](https://huggingface.co/docs/transformers/model_doc/llama2) on the [Europarl-ST dataset](https://huggingface.co/datasets/tj-solergibert/Europarl-ST), but only that [dataset of Europarl-ST focused on the text data for MT from English](https://huggingface.co/datasets/tj-solergibert/Europarl-ST-processed-mt-en).

In [3]:
# from datasets import load_dataset

# raw_datasets = load_dataset("tj-solergibert/Europarl-ST-processed-mt-en")

# print(raw_datasets)

from src.data import get_es_eo_dataset

raw_datasets = get_es_eo_dataset(CONFIG)

print(raw_datasets)

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 229673
    })
    test: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 28709
    })
    valid: Dataset({
        features: ['source_text', 'dest_text', 'dest_lang'],
        num_rows: 28709
    })
})


As shown, the Europarl-ST already comes with a pre-defined partition on the three conventional sets: training, validation and test. Each set is a dictionary with a list of source sentences (source_text), target sentences (dest_text) and the target language (dest_lang).

Let's take a closer look at the features of the training set:

In [4]:
raw_datasets["train"].features

{'source_text': Value('string'),
 'dest_text': Value('string'),
 'dest_lang': Value('int64')}

As you can see, the possible target languages are German, English, Spanish, French, Italian, Dutch, Polish, Portuguese and Romanian.

Let us take a look at the translations of the first two English sentences:

In [5]:
raw_datasets["train"][:14]["source_text"]

['ninguno de ellos verdad?.',
 'hermano y hermana, con toda tu fuerza de joven ¿que le contestas?”',
 'en 1870 la judería de roma, la última legal que quedaba en europa, fue abrogasea por víctor manuel ii, rey de italia.',
 'como hago el proceso de limpieza?',
 'no pongáis vuestro afecto en la soberanía mortal y no os regocijéis con ella.',
 'las máquinas de guerra.',
 'en algunos casos, podemos informar a su más reciente investigación de mercado y la información del producto.',
 'antes de que vayas...',
 'en 1762, rousseau publicó el contrato social principes du droit politique (en inglés, literalmente, del contrato social, principios de derecho político ) en abril.',
 'riesgos sobre clonación en 1996, fue clonada la oveja dolly.',
 'a diferencia de la “cooptación” estrategia descrita para las empresas, el oasd (ha) reconoce la neutralidad de la salud como un servicio esencial.',
 'luego suplicó al rey que no lo regresara a la casa de jonatán para no morir allí.',
 'gente a la que le 

In [6]:
raw_datasets["train"][:14]["dest_text"]

['– ĉu fakte neniu el ili?',
 'frato, fratino, kion vi al li en via forto de juneco respondos?',
 'en 1870 la juda kvartalo de romo, la lasta kiu restis en eŭropo, estis malaperigita de la reĝo viktoro emanuelo la 2-a, reĝo de italio.',
 'do, kiel la purigisto decidus?',
 'ne konante (not knowing ) la profundecon, ne iru en la riveron.',
 'la milita maŝino.',
 'en iuj kazoj, oni povas informi vian lastan merkato esplorado kaj produkto info.',
 'antaŭ vi iras ...',
 'en 1762, rousseau publikigis du contrat social, principes du rajtopolitikve (en la angla, laŭlitere of the social contract, principles of political right (de la socialkontrakto, principoj de politika rajto) ) en aprilo.',
 'per tiu metodo estis klonita en 1996 la ŝafo dolly.',
 'male al la "ko-opcio-" strategio priskribita por entreprenoj, la oasd (ha) rekonas la neŭtralecon de sano kiel esenca servo.',
 'mi humile petegis la reĝon, ke li ne revenigu min en la domon de jonatan, mi ne mortu tie.',
 'kiuj ŝatas poezion,',
 'm

In [7]:
raw_datasets["train"][:14]["dest_lang"]

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

As shown, each English sentence is repeated for each of the seven target languages (0: 'de', 2: 'es', 3: 'fr', 4: 'it', 5: 'nl', 6: 'pl', 7: 'pt').

The Llama2 model is a pretrained Large Language Model (LLM) ready to tackle several NLP tasks, being one of the them the translation from English into Spanish. Let us filter the Europarl-ST only for English into Spanish using a simple [lambda function](https://realpython.com/python-lambda/) with the [Dataset.filter() function](https://huggingface.co/docs/datasets/v2.9.0/en/package_reference/main_classes#datasets.Dataset.filter).

In [ ]:
# lang="es"
# lang_id = raw_datasets["train"].features["dest_lang"].names.index(lang)
# raw_datasets = raw_datasets.filter(lambda x: x["dest_lang"] == lang_id)

More precisely, we are going to be using the Llama-2 checkpoint [meta-llama/Llama-2-7b-hf](https://huggingface.co/meta-llama/Llama-2-7b-hf) to run our experiments for which you need to accept the LLAMA 2 COMMUNITY LICENSE AGREEMENT. Processing your request may take some time, so please do it in advance.

Logging in HuggingFace to be granted access to Llama2 with 7B parameters:

In [9]:
import os
import dotenv
from huggingface_hub import login

dotenv.load_dotenv()
login(token=os.getenv("HUGGING_FACE_TOKEN"))

python-dotenv could not parse statement starting at line 2
python-dotenv could not parse statement starting at line 4
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 8


We can apply the tokenizer function to any dataset taking advantage that Hugging Face Datasets are [Apache Arrow](https://arrow.apache.org) files stored on the disk, so you only keep the samples you ask for loaded in memory.

To keep the data as a dataset, we will use the [Dataset.map() function](https://huggingface.co/docs/datasets/en/package_reference/main_classes#datasets.Dataset.map). This also allows us some extra flexibility, if we need more preprocessing done than just tokenization. The map() method works by applying a function on each element of the dataset.

In our case, each sample pair is going to be preprocessed according to the needs of the model that is to be prompted. In the case of Llama2, it is recommended to explicitly state a task prompt for each source sentence:

In [ ]:
from transformers import AutoTokenizer

checkpoint = CONFIG.model_name #"meta-llama/Llama-2-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint,
    token=True,
    padding=True,
    pad_to_multiple_of=8,
    truncation=True,
    max_length=CONFIG.CONFIG.max_tok_length,
    padding_side='left',
    )
tokenizer.pad_token = tokenizer.eos_token

In [11]:
def preprocess_function(sample):
    model_inputs = tokenizer(
        sample["source_text"], 
        text_target = sample["dest_text"],
        )
    return model_inputs

The way the Datasets library applies this processing is by adding new fields to the datasets, one for each key in the dictionary returned by the tokenize function, that is, *input_ids*, *attention_mask* and *labels*. We can check what the preprocess_function is doing with a small sample

In [12]:
sample = raw_datasets["train"].select(range(2))
model_input = preprocess_function({
    "source_text": list(sample["source_text"]),
    "dest_text": list(sample["dest_text"]),
})
print(model_input)

{'input_ids': [[1, 26511, 9447, 316, 21135, 12868, 328, 9808], [1, 18606, 1562, 343, 18606, 1648, 29892, 378, 20223, 5291, 8470, 1362, 316, 432, 9813, 18613, 802, 454, 17793, 294, 6677]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], 'labels': [[1, 785, 29871, 31431, 29884, 285, 17230, 302, 264, 5871, 560, 29040, 29973], [1, 1424, 1219, 29892, 20998, 1789, 29892, 413, 291, 3516, 394, 619, 427, 3025, 363, 517, 316, 4707, 687, 29877, 10049, 359, 29973]]}


In [13]:
for sample in model_input['input_ids']:
    print(tokenizer.convert_ids_to_tokens(sample))

['<s>', '▁ning', 'uno', '▁de', '▁ellos', '▁verd', 'ad', '?.']
['<s>', '▁herm', 'ano', '▁y', '▁herm', 'ana', ',', '▁con', '▁toda', '▁tu', '▁fuer', 'za', '▁de', '▁j', 'oven', '▁¿', 'que', '▁le', '▁contest', 'as', '?”']


We can recover the source text by applying [batch_decode](https://huggingface.co/docs/transformers/en/internal/tokenization_utils#transformers.PreTrainedTokenizerBase.batch_decode) of the tokenizer 

In [14]:
tokenizer.batch_decode(model_input['input_ids'])

['<s> ninguno de ellos verdad?.',
 '<s> hermano y hermana, con toda tu fuerza de joven ¿que le contestas?”']

Now, we can apply the preprocess_function to the raw datasets (training, validation and test):

In [15]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

Map: 100%|██████████| 28709/28709 [00:00<00:00, 38530.73 examples/s]


We are going to filter the tokenized datasets by maximum number of tokens in source and target language:

In [ ]:
tokenized_datasets = tokenized_datasets.filter(lambda x: len(x["input_ids"]) <= CONFIG.CONFIG.max_tok_length and len(x["labels"]) <= CONFIG.CONFIG.max_tok_length , desc=f"Discarding source and target sentences with more than {CONFIG.CONFIG.max_tok_length} tokens")

Discarding source and target sentences with more than 16 tokens:   0%|          | 0/229673 [00:00<?, ? examples/s]

Discarding source and target sentences with more than 16 tokens: 100%|██████████| 229673/229673 [00:03<00:00, 65329.24 examples/s]
Discarding source and target sentences with more than 16 tokens: 100%|██████████| 28709/28709 [00:00<00:00, 50126.94 examples/s]
Discarding source and target sentences with more than 16 tokens: 100%|██████████| 28709/28709 [00:00<00:00, 65496.43 examples/s]


We can take a quick look at the length histogram in the source language:

In [ ]:
dic = {}
for sample in tokenized_datasets['train']:
    sample_length = len(sample['input_ids'])
    if sample_length not in dic:
        dic[sample_length] = 1
    else:
        dic[sample_length] += 1 

for i in range(1,CONFIG.CONFIG.max_tok_length+1):
    if i in dic:
        print(f"{i:>2} {dic[i]:>3}")

 3  24
 4 269
 5 1232
 6 3298
 7 6451
 8 9513
 9 11170
10 11145
11 9838
12 8017
13 5965
14 4126
15 2711
16 1681


Checking a sample after filtering by maximum number of tokens:

In [18]:
for sample in tokenized_datasets['train'].select(range(5)):
    print(sample['input_ids'])
    print(sample['attention_mask'])
    print(sample['labels'])

[1, 26511, 9447, 316, 21135, 12868, 328, 9808]
[1, 1, 1, 1, 1, 1, 1, 1]
[1, 785, 29871, 31431, 29884, 285, 17230, 302, 264, 5871, 560, 29040, 29973]
[1, 1986, 298, 4425, 560, 14177, 29877, 316, 2485, 12343, 1362, 29973]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[1, 437, 29892, 413, 709, 425, 3708, 335, 5137, 18937, 375, 29973]
[1, 1869, 10269, 339, 10189, 316, 9798, 29889]
[1, 1, 1, 1, 1, 1, 1, 1]
[1, 425, 2316, 2028, 611, 31805, 1789, 29889]
[1, 12971, 316, 712, 325, 388, 294, 856]
[1, 1, 1, 1, 1, 1, 1, 1]
[1, 385, 941, 30520, 3516, 3805, 294, 2023]
[1, 330, 2016, 263, 425, 712, 454, 330, 29664, 425, 772, 267, 1553, 29889]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[1, 413, 5871, 29926, 29871, 31805, 271, 294, 772, 6096, 291, 29892]


In [ ]:
src = "en"
tgt = lang
task_prefix = f"Translate from {src} to {tgt}:\n"
shots = ""
s = ""

prefix_tok_len = len(tokenizer.encode(f"{task_prefix}{shots}{src}: {s} = {tgt}: "))
shot_tok_len   = len(tokenizer.encode(f"{src}: {s} = {tgt}: {s}\n"))
max_tok_len = prefix_tok_len
max_tok_len += CONFIG.num_shots * (shot_tok_len + 2 * CONFIG.CONFIG.max_tok_length) 
max_tok_len += CONFIG.CONFIG.max_tok_length

random_seed = 13
sample = tokenized_datasets['train'].shuffle(seed=random_seed).select(range(CONFIG.num_shots))
for s in sample: shots += f"{src}: {s['source_text']} = {tgt}: {s['dest_text']}\n" 

def preprocess4test_function(sample):
    inputs = [f"{task_prefix}{shots}{src}: {s} = {tgt}: " for s in sample["source_text"]]
    model_inputs = tokenizer(
        inputs,
        max_length=max_tok_len, 
        truncation=True, 
        return_tensors="pt", 
        padding=True)
    return model_inputs

The way the Datasets library applies this processing is by adding new fields to the datasets, one for each key in the dictionary returned by the tokenize function, that is, *input_ids*, *attention_mask* and *labels*:

In [20]:
sample = tokenized_datasets['test'].select(range(5))
model_input = preprocess4test_function(sample)
print(model_input)
print(tokenizer.batch_decode(model_input['input_ids']))

{'input_ids': tensor([[    1,  4103,  9632,   515,   427,   304,   831, 29901,    13,   264,
         29901,   633,  1569,   432,  2002,   443, 19368,  5112,   342,  1789,
           316,   443,  2368,   353,   831, 29901,   432,  2002, 14079, 11088,
           316,   425, 29871, 31805, 29873,  1219,   316,  5112,   342,  1789,
            13,   264, 29901,   633,   307,  1232,   288, 14736, 29892,   553,
         29880,   398,  1182,   912, 29889,   353,   831, 29901, 29871],
        [    2,     2,     2,     1,  4103,  9632,   515,   427,   304,   831,
         29901,    13,   264, 29901,   633,  1569,   432,  2002,   443, 19368,
          5112,   342,  1789,   316,   443,  2368,   353,   831, 29901,   432,
          2002, 14079, 11088,   316,   425, 29871, 31805, 29873,  1219,   316,
          5112,   342,  1789,    13,   264, 29901, 21135,  5821,   381,  1715,
           425,   907,   655,  1290, 29889,   353,   831, 29901, 29871],
        [    1,  4103,  9632,   515,   427,   304,

In [21]:
preprocessed_test_dataset = tokenized_datasets['test'].map(preprocess4test_function, batched=True)

Map: 100%|██████████| 9593/9593 [00:00<00:00, 29059.27 examples/s]


In [22]:
for sample in preprocessed_test_dataset.select(range(5)):
    print(sample['input_ids'])
    print(sample['attention_mask'])
    print(sample['labels'])

[2, 2, 2, 1, 4103, 9632, 515, 427, 304, 831, 29901, 13, 264, 29901, 633, 1569, 432, 2002, 443, 19368, 5112, 342, 1789, 316, 443, 2368, 353, 831, 29901, 432, 2002, 14079, 11088, 316, 425, 29871, 31805, 29873, 1219, 316, 5112, 342, 1789, 13, 264, 29901, 633, 307, 1232, 288, 14736, 29892, 553, 29880, 398, 1182, 912, 29889, 353, 831, 29901, 29871]
[0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[1, 3737, 4439, 571, 8247, 3431, 352, 3848, 29876, 29892, 413, 2156, 595, 3922, 29889]
[2, 2, 2, 2, 2, 2, 1, 4103, 9632, 515, 427, 304, 831, 29901, 13, 264, 29901, 633, 1569, 432, 2002, 443, 19368, 5112, 342, 1789, 316, 443, 2368, 353, 831, 29901, 432, 2002, 14079, 11088, 316, 425, 29871, 31805, 29873, 1219, 316, 5112, 342, 1789, 13, 264, 29901, 21135, 5821, 381, 1715, 425, 907, 655, 1290, 29889, 353, 831, 29901, 29871]
[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 

bitsandbytes is a quantization library with a Transformers integration. With this integration, you can quantize a model to 8 or 4-bits and enable many other options by configuring the BitsAndBytesConfig class. For example, you can:

<ul>
<li>set load_in_4bit=True to quantize the model to 4-bits when you load it</li>
<li>set bnb_4bit_quant_type="nf4" to use a special 4-bit data type for weights initialized from a normal distribution</li>
<li>set bnb_4bit_use_double_quant=True to use a nested quantization scheme to quantize the already quantized weights</li>
<li>set bnb_4bit_compute_dtype=torch.bfloat16 to use bfloat16 for faster computation</li>
</ul>


In [23]:
import torch
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

Pass the quantization_config to the from_pretrained method.

In [24]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    checkpoint,
    token=True,
    quantization_config=quantization_config,
    dtype=torch.bfloat16,
)


Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.37s/it]


# Inference

Loading default inference parameters for the model, so that additional parameters could be added and passed to the [generate function](https://huggingface.co/docs/transformers/main_classes/text_generation):

In [25]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(
    checkpoint,
    )

print(generation_config)

GenerationConfig {
  "bos_token_id": 1,
  "do_sample": true,
  "eos_token_id": 2,
  "max_length": 4096,
  "pad_token_id": 0,
  "temperature": 0.6,
  "top_p": 0.9
}



As observed, the default search strategy for Llama-2 is Top-p with probability 0.9 and temperature 0.6 ($0<T<1$ amplifies output probability differences and makes output more deterministic). [The search strategy can be selected](https://huggingface.co/docs/transformers/en/generation_strategies) at inference time. 

First, the test set is divided in small batches to reduce GPU memory comsumption:

In [ ]:
batch_tokenized_test = preprocessed_test_dataset.batch(CONFIG.batch_size)

Batching examples: 100%|██████████| 9593/9593 [00:00<00:00, 28167.64 examples/s]


In [28]:
import tqdm 

number_of_batches = len(batch_tokenized_test["input_ids"])
output_sequences = []
for i in tqdm.tqdm(range(number_of_batches)):
    with torch.no_grad():
        output_batch = model.generate(
            generation_config=generation_config, 
            input_ids=torch.tensor(batch_tokenized_test["input_ids"][i]).cuda(), 
            attention_mask=torch.tensor(batch_tokenized_test["attention_mask"][i]).cuda(), 
            max_length = max_tok_len, 
            num_beams=1, 
            do_sample=False,
        )
    output_sequences.extend(output_batch)


  0%|          | 0/300 [00:00<?, ?it/s]

100%|██████████| 300/300 [06:00<00:00,  1.20s/it]


## Evaluation

The output of the model is automatically evaluated compared to the reference translations. To this purpose, we use the [Evaluate library](https://huggingface.co/docs/evaluate) which includes the definition of generic and task-specific metrics. In our case, we use the [BLEU metric](https://huggingface.co/spaces/evaluate-metric/bleu), or to be more precise, [sacreBLEU](https://huggingface.co/spaces/evaluate-metric/sacrebleu).

In [29]:
from evaluate import load

metric_bleu = load("sacrebleu")
metric_comet = load("comet")

/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 107546.26it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/home/turbotowerlnx/Documents/Master/TA/TA-Spanish-Esperanto-Translator/venv/lib/python3.12/site-packages/pytorch_lig

The example below performs a basic post-processing to decode the predictions and extract the translation:

In [37]:
import re

def compute_metrics(sample, output_sequences):
    inputs = [f"{task_prefix}{shots}{src}: {s} = {tgt}: " for s in sample["source_text"]]
    preds = tokenizer.batch_decode(output_sequences, skip_special_tokens=True)
    # print(inputs)
    # print(preds)
    for i, (input,pred) in enumerate(zip(inputs,preds)):
      pred = re.search(r'^.*\n',pred.removeprefix(input).lstrip())
      if pred is not None:
        preds[i] = pred.group()[:-1]
      else:
        preds[i] = ""
    # print(sample["source_text"])
    # print(sample["dest_text"])
    # print(preds)
    result_bleu = metric_bleu.compute(
       predictions=preds, 
       references=sample["dest_text"]
    )
    result_comet = metric_comet.compute(
        sources=sample["source_text"],
        predictions=preds, 
        references=sample["dest_text"]
    )
    result = {
      "bleu": result_bleu["score"],
      "comet": result_comet["mean_score"]
      }
    return result

In [ ]:
result = compute_metrics(preprocessed_test_dataset,output_sequences, )
print(f'BLEU score: {result["bleu"]:0.4f}')
print(f'COMET score: {result["comet"]:0.4f}')

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the f

BLEU score: 0.626339085131259
COMET score: 0.5475572593409961
